# Урок 11. ООП, часть 2: основные принципы
**Python для анализа данных · CyberFormula3 × TechOrda**

| Время | Блок |
|---|---|
| 0–15 | Разминка «что выведет код» · Повторение · Практика 0 |
| 15–35 | Наследование и `super()` |
| 35–45 | Инкапсуляция и `@property` |
| 45–60 | Полиморфизм и магические методы |
| 60–75 | Практика 1: всё по теме |
| 75–87 | Практика 2: закрепление — `Animal` → `Dog`, `Cat` |
| 87–90 | Итоги |

**Правило занятия:** предскажи → запусти → объясни.

**Уровни заданий:** **[База]** — должен сделать каждый · **[Челлендж]** — чуть сложнее · **[Про]** — для тех, кто всё успел.

## 📖 Справочник урока
Держите открытым весь урок: здесь всё, что понадобится сегодня.

### 1. Что мы уже знаем (урок 10)

| Понятие | Как пишется | Пример |
|---|---|---|
| Класс | `class Имя:` | `class Student:` |
| Создание объекта | `x = Класс(...)` | `a = Student("Аружан", "DA-01")` |
| `__init__` | вызывается сам при создании | `def __init__(self, name):` |
| `self` | «этот объект» | `self.name = name` |
| Атрибут | **без скобок** | `a.name` |
| Метод | **со скобками** | `a.average()` |

### 2. Новое: три принципа ООП

| Принцип | Простыми словами | Как пишется |
|---|---|---|
| **Наследование** | новый класс берёт всё у старого и добавляет своё | `class Dog(Animal):` |
| `super()` | «обратись к родителю» | `super().__init__(name, age)` |
| **Переопределение** | у ребёнка свой вариант метода родителя | в `Dog` снова пишем `def speak(self):` |
| **Инкапсуляция** | данные защищены, менять — только через правила | `self._balance` + `@property` |
| `@property` | метод, который читается как атрибут | `acc.balance` — **без скобок** |
| `@имя.setter` | проверка при изменении атрибута | `p.price = -5` → ошибка |
| **Полиморфизм** | один вызов — разное поведение у разных объектов | `for a in animals: a.speak()` |
| `isinstance(x, Класс)` | является ли объект этим классом (или его потомком) | `isinstance(dog, Animal)` → `True` |

### 3. Магические методы — «научить объект понимать Python»

| Метод | Срабатывает, когда пишем | Должен вернуть |
|---|---|---|
| `__str__(self)` | `print(obj)`, `str(obj)`, f-строка | **строку** |
| `__len__(self)` | `len(obj)` | целое число |
| `__eq__(self, other)` | `obj1 == obj2` | `True` / `False` |
| `__lt__(self, other)` | `obj1 < obj2`, а значит `sorted`, `min`, `max` | `True` / `False` |

**Шаблон наследования:**
```python
class Account:                                  # родитель
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

class SavingsAccount(Account):                  # ребёнок: (Родитель)
    def __init__(self, owner, balance=0, rate=0.14):
        super().__init__(owner, balance)        # родитель сохранит owner и balance
        self.rate = rate                        # своё, новое

    def add_interest(self):                     # новый метод
        self.balance = self.balance + self.balance * self.rate
```

### 4. Типичные ошибки

| Симптом | Причина | Как исправить |
|---|---|---|
| `AttributeError: ... has no attribute 'name'` у ребёнка | в `__init__` ребёнка забыли `super().__init__(...)` | первой строкой: `super().__init__(name, age)` |
| `TypeError: __init__() missing ... argument` | в `super().__init__()` передали не все данные родителя | передайте всё, что просит `__init__` родителя |
| `TypeError: 'int' object is not callable` | вызвали свойство со скобками: `acc.balance()` | `@property` читается **без скобок**: `acc.balance` |
| `AttributeError: property ... has no setter` | пытаемся изменить свойство без сеттера | добавить `@balance.setter` или менять через метод |
| `RecursionError` в сеттере | внутри сеттера написали `self.price = ...` вместо `self._price = ...` | внутри — всегда с подчёркиванием `_` |
| `TypeError: __str__ returned non-string` | `__str__` что-то печатает, но не возвращает строку | `return f"..."`, а не `print(...)` |
| `print(obj)` → `<__main__.X object at 0x...>` | у класса нет `__str__` | добавить `def __str__(self): return ...` |

In [19]:
class Box:
    def __init__(self, n):
        self.n = n

    def double(self):
        self.n = self.n * 2

In [ ]:
b = Box(3)
b.double()
b.double()
print(b.n)

In [ ]:
a = Box(5)
c = a
c.n = 100
print(a.n)

In [ ]:
boxes = [Box(1), Box(2), Box(3)]
print(sum(x.n for x in boxes))

## 0. Повторение: классы и объекты (урок 10)
Вспомним класс `Student` — сегодня мы будем его **расширять**.

In [1]:
class Student:
    def __init__(self, name, group):
        self.name = name
        self.group = group
        self.grades = []

    def add_grade(self, grade):
        if 0 <= grade <= 100:
            self.grades.append(grade)

    def average(self):
        if len(self.grades) == 0:
            return 0
        return round(sum(self.grades) / len(self.grades), 1)


aruzhan = Student("Аружан", "DA-01")
for g in [92, 88, 95]:
    aruzhan.add_grade(g)
print(aruzhan.name, aruzhan.grades, aruzhan.average())

Аружан [92, 88, 95] 91.7


In [2]:
# ПРЕДСКАЖИТЕ: что напечатается?
print(aruzhan)

> Запомните это странное `<__main__.Student object at 0x...>`. Через 40 минут мы научим `print` показывать студента по-человечески.

### Практика 0 (5 мин): повторение, по одной строке на задачу
**[База]**
1. Создайте студента `d`: «Данияр», группа «DA-02». 2. Добавьте ему оценку 70. 3. Выведите его средний балл.

**[Челлендж]**

4. Выведите, сколько оценок у `aruzhan`. 5. Одной строкой выведите, у кого из двоих (`aruzhan` или `d`) средний балл выше — имя.

**[Про]**

6. Одной строкой создайте список `group` из `aruzhan` и `d` и выведите список их имён.

### Что неудобно?
Нужен ещё класс **преподавателя**. У него тоже есть имя и группа, но вместо оценок — предмет и стаж.
```python
class Teacher:
    def __init__(self, name, group, subject):
        self.name = name          # ← копия из Student
        self.group = group        # ← копия из Student
        self.subject = subject
```
Если завтра у всех людей появится `email` — придётся исправлять **каждый** класс. Копирование кода = ошибки.

> **Наследование** решает это: общее пишем **один раз** в классе-родителе, а дети берут его себе и добавляют своё.

---
## 1. Наследование и `super()`

| Понятие | Что это | Пример |
|---|---|---|
| **Родитель** (базовый класс) | общий шаблон | `Person` — у любого человека есть имя и возраст |
| **Ребёнок** (наследник) | берёт всё у родителя + своё | `Student(Person)` — ещё группа и оценки |
| **Переопределение** | у ребёнка свой вариант метода | `describe()` у студента рассказывает про группу |

### Шаг 1. 

In [34]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def describe(self):
        return f"{self.name}, {self.age} лет"


class Student(Person):     # в скобках — родитель
    pass                   # ничего своего не пишем!


In [ ]:
s = Student("Аружан", 19)
print(s.name)              # ПРЕДСКАЖИТЕ: откуда у Student атрибут name?
print(s.describe())

### Шаг 2. Своё + родительское: `super().__init__()`
Студенту нужна ещё группа. Пишем свой `__init__`, но **имя и возраст отдаём родителю** через `super()` — «обратись к родителю».

In [37]:
class Student(Person):
    def __init__(self, name, age, group):
        super().__init__(name, age)    # родитель сохранит name и age
        self.group = group             # своё

    def is_adult(self):                # новый метод — только у студентов
        return self.age >= 18



In [38]:
s = Student("Аружан", 19, "DA-01")
print(s.name, s.age, s.group)
print(s.describe())        # метод родителя
print(s.is_adult())        # свой метод

Аружан 19 DA-01
Аружан, 19 лет
True


### Шаг 3. Переопределение: свой вариант метода
Если у ребёнка написать метод **с тем же именем**, Python возьмёт его, а не родительский. Через `super()` можно позвать и родительскую версию.

In [41]:
class Student(Person):
    def __init__(self, name, age, group):
        super().__init__(name, age)
        self.group = group

    def describe(self):                          # переопределили
        base = super().describe()                # "Аружан, 19 лет" — от родителя
        return f"{base}, студент группы {self.group}"


class Teacher(Person):
    def __init__(self, name, age, subject):
        super().__init__(name, age)
        self.subject = subject

    def describe(self):
        return f"{super().describe()}, преподаёт {self.subject}"


p = Person("Гульшат", 30)
s = Student("Аружан", 19, "DA-01")
t = Teacher("Ерлан", 35, "Python")
     # ПРЕДСКАЖИТЕ все три строки до запуска

In [ ]:
print(p.describe())
print(s.describe())
print(t.describe()) 

In [ ]:
# isinstance — проверка «является ли объект этим классом или его потомком»
print(isinstance(s, Student))   # ПРЕДСКАЖИТЕ
print(isinstance(s, Person))    # ПРЕДСКАЖИТЕ
print(isinstance(p, Student))   # ПРЕДСКАЖИТЕ

### Пример из аналитики: счета в банке
Базовый счёт из урока 10 + два «вида» счёта, которые его наследуют.

In [7]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount > 0:
            self.balance = self.balance + amount

    def withdraw(self, amount):
        if amount > self.balance:
            print(f"{self.owner}: недостаточно средств")
        else:
            self.balance = self.balance - amount


class SavingsAccount(Account):                    # депозит: начисляет проценты
    def __init__(self, owner, balance=0, rate=0.14):
        super().__init__(owner, balance)
        self.rate = rate

    def add_interest(self):
        self.balance = self.balance + self.balance * self.rate


class CreditAccount(Account):                     # кредитка: можно уйти в минус до лимита
    def __init__(self, owner, balance=0, limit=500_000):
        super().__init__(owner, balance)
        self.limit = limit

    def withdraw(self, amount):                   # переопределили правило снятия
        if self.balance - amount < -self.limit:
            print(f"{self.owner}: превышен кредитный лимит")
        else:
            self.balance = self.balance - amount

In [ ]:
dep = SavingsAccount("Мадина", 1_000_000)
dep.add_interest()
print(dep.balance)                 # ПРЕДСКАЖИТЕ

In [ ]:
card = CreditAccount("Данияр", 100_000)
card.withdraw(300_000)             # ПРЕДСКАЖИТЕ: пройдёт ли?
print(card.balance)
card.withdraw(400_000)             # а это?
print(card.balance)

### Найдите ошибки — ПРЕДСКАЖИТЕ, что будет

In [ ]:
class Teacher(Person):
    def __init__(self, name, age, subject):
        self.subject = subject

t = Teacher("Ерлан", 35, "Python")
print(t.name)

In [ ]:
class Teacher(Person):
    def __init__(self, name, age, subject):
        super().__init__(name)
        self.subject = subject

t = Teacher("Ерлан", 35, "Python")

---
## 2. Инкапсуляция и `@property`

### Проблема: данные никто не охраняет

In [ ]:
acc = Account("Аружан", 50_000)
acc.balance = -1_000_000     # ПРЕДСКАЖИТЕ: Python разрешит?
print(acc.balance)

Метод `withdraw` проверяет баланс, но его можно **обойти**: записать в атрибут что угодно напрямую. В аналитике это значит — «грязные» данные внутри объекта.

**Инкапсуляция** = данные лежат внутри объекта, а менять их можно только по правилам.

| Запись | Смысл |
|---|---|
| `self.balance` | открытый атрибут — меняй как хочешь |
| `self._balance` | **соглашение**: «внутреннее, снаружи не трогать» (Python не запрещает, но программисты договорились) |
| `@property` | метод, который **читается как атрибут** — без скобок |
| `@balance.setter` | проверка, которая срабатывает при `obj.balance = ...` |

### Шаг 1. `@property` — только чтение
Назовём новую версию счёта `SafeAccount`, чтобы не путать с `Account` из блока наследования.

In [75]:
class SafeAccount:          # защищённый счёт
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance            # «внутренний» атрибут

    @property
    def balance(self):                     # читаем как атрибут: acc.balance
        return self._balance

    def deposit(self, amount):
        if amount > 0:
            self._balance = self._balance + amount

    def withdraw(self, amount):
        if amount > self._balance:
            print("Недостаточно средств")
        else:
            self._balance = self._balance - amount


acc = SafeAccount("Аружан", 50_000)
print(acc.balance)          # без скобок!
acc.deposit(10_000)
print(acc.balance)

50000
60000


In [ ]:
# ПРЕДСКАЖИТЕ: что будет?
acc.balance = 1_000_000

Теперь баланс меняется **только** через `deposit` и `withdraw`. Записать его напрямую нельзя.

### Шаг 2. `setter` — разрешить менять, но с проверкой
Цена товара должна меняться (скидки!), но не может быть отрицательной.

In [91]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price                 # ← это уже вызывает сеттер ниже!

    @property
    def price(self):
        return self._price
    
    @price.setter
    def price(self, value):
        if value < 0:
            print(f"Цена не может быть отрицательной: {value}")
        else:
            self._price = value

In [ ]:
p = Product("Ноутбук", 350_000)
p.price = 320_000          # скидка — разрешено
print(p.price)
p.price = -5               # ПРЕДСКАЖИТЕ
print(p.price)             # ПРЕДСКАЖИТЕ: какая цена осталась?

### Вычисляемое свойство
`@property` удобно и для значений, которые **считаются** из других данных, — как колонка-формула в Excel.

In [94]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height



In [ ]:
r = Rectangle(4, 5)
print(r.area)          # 20 — без скобок
r.width = 10
print(r.area)          # ПРЕДСКАЖИТЕ: пересчитается?

---
## 3. Полиморфизм и магические методы

### Полиморфизм: один вызов — разное поведение
**Полиморфизм** — разные классы имеют метод **с одним и тем же именем**, и мы вызываем его, не думая, какой перед нами объект. Каждый объект сам знает, как ему это сделать.

In [101]:
class Shape:
    def area(self):
        return 0

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return round(3.14 * self.r ** 2, 2)

class Square(Rectangle):              # квадрат — это прямоугольник с равными сторонами
    def __init__(self, side):
        super().__init__(side, side)


shapes = [Rectangle(4, 5), Circle(2), Square(3)]
for sh in shapes:
    print(type(sh).__name__, sh.area())   # один и тот же вызов area() — три разных формулы

print("Общая площадь:", sum(sh.area() for sh in shapes))

Rectangle 20
Circle 12.56
Square 9
Общая площадь: 41.56


> Цикл **не проверяет**, кто перед ним — круг или квадрат. Он просто зовёт `area()`. Добавим завтра `Triangle` с методом `area()` — цикл менять не придётся. Вот зачем полиморфизм.

### Магические методы: учим объект понимать Python
Помните `print(aruzhan)` → `<__main__.Student object at 0x...>`? Python не знает, как показать наш объект. **Магические методы** (с двумя подчёркиваниями) объясняют ему это.

In [ ]:
class Student:
    def __init__(self, name, group):
        self.name = name
        self.group = group
        self.grades = []

    def add_grade(self, grade):
        if 0 <= grade <= 100:
            self.grades.append(grade)

    def average(self):
        if len(self.grades) == 0:
            return 0
        return round(sum(self.grades) / len(self.grades), 1)

    def __str__(self):                     # print(student)
        return f"{self.name} ({self.group}), средний балл {self.average()}"

    def __len__(self):                     # len(student) — сколько оценок
        return len(self.grades)

    def __eq__(self, other):               # student1 == student2
        return self.name == other.name and self.group == other.group

    def __lt__(self, other):               # student1 < student2 — сравниваем по среднему баллу
        return self.average() < other.average()

group = [Student("Аружан", "DA-01"), Student("Данияр", "DA-01"),
         Student("Мадина", "DA-02"), Student("Ерлан", "DA-02")]
all_grades = [[92, 88, 95], [64, 70, 58], [97, 91, 99], [75, 80, 72]]
for st, grades in zip(group, all_grades):
    for g in grades:
        st.add_grade(g)

In [106]:
print(group[0])             # ПРЕДСКАЖИТЕ — теперь работает __str__
print(len(group[0]))        # __len__
print(group[0] == Student("Аружан", "DA-01"))   # __eq__
print(group[0] < group[2])  # __lt__: Аружан учится хуже Мадины?

Аружан (DA-01), средний балл 91.7
3
True
True


**Бонус `__lt__`:** раз Python умеет сравнивать студентов, `sorted`, `min` и `max` работают **без `key=lambda`**:

In [105]:
print(max(group))                  # лучший студент — сразу по-человечески
print(min(group).name)             # имя худшего

for st in sorted(group, reverse=True):   # рейтинг
    print(st)

Мадина (DA-02), средний балл 95.7
Данияр
Мадина (DA-02), средний балл 95.7
Аружан (DA-01), средний балл 91.7
Ерлан (DA-02), средний балл 75.7
Данияр (DA-01), средний балл 64.0


| Было в уроке 10 | Стало |
|---|---|
| `print(st.name, st.average())` | `print(st)` |
| `len(st.grades)` | `len(st)` |
| `max(group, key=lambda st: st.average()).name` | `max(group).name` |

### Найдите ошибки — ПРЕДСКАЖИТЕ, что будет

In [ ]:
class City:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        print(f"Город {self.name}")

print(City("Алматы",))

---
## Практика 1 (15 мин) — по одной строке на задачу
Используйте всё, что создали выше: счета (`SavingsAccount`, `CreditAccount`), `Product`, `Student` с магическими методами и список `group`, фигуры `shapes`.

**[База]**
1. Создайте депозит `my_dep` на своё имя: баланс 500 000, ставка 0.16. 2. Начислите проценты и выведите баланс. 3. Выведите `print` второго студента из `group`. 4. Создайте товар `phone` за 189 000 ₸ и попробуйте поставить ему цену `-100`. Какая цена осталась?

**[Челлендж]**

5. Одной строкой проверьте: является ли `my_dep` объектом `Account`? А объектом `CreditAccount`? 6. Выведите имена студентов `group` в порядке рейтинга — от лучшего к худшему (без `lambda`!).

**[Про]**

7. Одной строкой: сколько фигур в `shapes` — прямоугольники, **включая** квадраты? 8. Одной строкой: средний балл всей группы (среднее средних, округлить до 1 знака).



**Ответьте устно:**
1. Зачем нужно наследование? Что даёт `super().__init__(...)`?
2. Чем `self._balance` отличается от `self.balance`? Зачем `@property`?
3. Что такое полиморфизм? Приведите пример из сегодняшнего урока.
4. Что нужно добавить в класс, чтобы `print(obj)` показывал понятный текст? А чтобы работал `sorted(list_of_obj)`?